In [14]:
from ..Agent import *

load_dotenv(override=True)
#必须关闭思考模式，deepseek默认为思考模式，思考模式不支持结构化输出
#因为deepseek底层使用tool_choice作为伪工具传递结构化输出，思考模式不支持tool_choice
DEEPSEEK_API_KEY=os.getenv('DEEPSEEK_API_KEY')
DEEPSEEK_BASE_URL=os.getenv('DEEPSEEK_BASE_URL')
model=init_chat_model(
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    model='deepseek-v4-flash',
    model_provider='deepseek',
    extra_body={"thinking":{"type":"disabled"}}
)

class personInfo(BaseModel):
    name:str=Field(None,description='姓名')
    email:str=Field(None,description='个人邮箱')
class event(BaseModel):
    name:str=Field(None,description='活动名称')
    address:str=Field(None,description='活动地点')

def custom_error_call(error:Exception):
    error_str=str(error)
    print(f'捕获到错误类型{type(error).__name__}')
    print(f'错误{error_str}')
    if isinstance(error,StructuredOutputValidationError):
        return '字段格式错误，请检查输出格式是否匹配'
    elif isinstance(error,MultipleStructuredOutputsError):
        return '返回了多个结构化响应，请选择最相关的一个返回'
    else:
        return 'Error:{}'.format(error_str)

In [15]:


#使用tool_message_content自定义工具的返回消息而不是结构化消息，结构化输出只存储在structure_response中不作为消息返回
#handel_errors为True时由langchain的错误消息模版提醒大模型重试，false时抛异常，也可以自定义错误消息文本以及回调函数
#MultipleStructuredOutputsError和StructuredOutputValidationError分别是返回多个结构化输出和格式错误的异常
myagent=create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[personInfo,event],
        tool_message_content='提取完成',
        handle_errors=custom_error_call,
    ),
)
messages=HumanMessage('请提取以下信息作为结构化输出：张三，个人邮箱为zs@111.com，他到音乐厅参加了演唱会')
#也可以使用stream模式
response=myagent.invoke({
    'messages':messages,
})
rprint(response['messages'])

捕获到错误类型MultipleStructuredOutputsError
错误Model incorrectly returned multiple structured responses (personInfo, event) when only one is expected.


[
    HumanMessage(
        content='请提取以下信息作为结构化输出：张三，个人邮箱为zs@111.com，他到音乐厅参加了演唱会',
        additional_kwargs={},
        response_metadata={},
        id='0a73edcb-4335-4b9b-aa99-8f68bad46c08'
    ),
    AIMessage(
        content='',
        additional_kwargs={'refusal': None},
        response_metadata={
            'token_usage': {
                'completion_tokens': 101,
                'prompt_tokens': 387,
                'total_tokens': 488,
                'completion_tokens_details': None,
                'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384},
                'prompt_cache_hit_tokens': 384,
                'prompt_cache_miss_tokens': 3
            },
            'model_provider': 'deepseek',
            'model_name': 'deepseek-v4-flash',
            'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
            'id': 'dfc20af3-7b48-46d3-8101-66e04c6508c7',
            'finish_reason': 'tool_calls',
            'logprobs': None
        },
        id='lc_run--019f8788-92c5-7690-a18b-ca994bb84f6d-0',
        tool_calls=[
            {
                'name': 'personInfo',
                'args': {'name': '张三', 'email': 'zs@111.com'},
                'id': 'call_00_EZeYXMdIYG6pxlxvE4Rn0761',
                'type': 'tool_call'
            },
            {
                'name': 'event',
                'args': {'name': '演唱会', 'address': '音乐厅'},
                'id': 'call_01_OA4kQFHn3x6Sl2aVGVFn7147',
                'type': 'tool_call'
            }
        ],
        invalid_tool_calls=[],
        usage_metadata={
            'input_tokens': 387,
            'output_tokens': 101,
            'total_tokens': 488,
            'input_token_details': {'cache_read': 384},
            'output_token_details': {}
        }
    ),
    ToolMessage(
        content='返回了多个结构化响应，请选择最相关的一个返回',
        name='personInfo',
        id='a0bc0bfa-9bff-4caa-acd1-b972c88b90d9',
        tool_call_id='call_00_EZeYXMdIYG6pxlxvE4Rn0761'
    ),
    ToolMessage(
        content='返回了多个结构化响应，请选择最相关的一个返回',
        name='event',
        id='f8a1e9f5-8f08-4ed1-aa5c-1193f1a544b9',
        tool_call_id='call_01_OA4kQFHn3x6Sl2aVGVFn7147'
    ),
    AIMessage(
        content='',
        additional_kwargs={'refusal': None},
        response_metadata={
            'token_usage': {
                'completion_tokens': 55,
                'prompt_tokens': 539,
                'total_tokens': 594,
                'completion_tokens_details': None,
                'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 512},
                'prompt_cache_hit_tokens': 512,
                'prompt_cache_miss_tokens': 27
            },
            'model_provider': 'deepseek',
            'model_name': 'deepseek-v4-flash',
            'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
            'id': 'e0bf6f97-4bd3-4f02-ae3a-924de7271cc0',
            'finish_reason': 'tool_calls',
            'logprobs': None
        },
        id='lc_run--019f8788-9895-7d63-ab60-b91d3e654003-0',
        tool_calls=[
            {
                'name': 'personInfo',
                'args': {'email': 'zs@111.com', 'name': '张三'},
                'id': 'call_00_8gdxbIVpvChlvHDtoDg64917',
                'type': 'tool_call'
            }
        ],
        invalid_tool_calls=[],
        usage_metadata={
            'input_tokens': 539,
            'output_tokens': 55,
            'total_tokens': 594,
            'input_token_details': {'cache_read': 512},
            'output_token_details': {}
        }
    ),
    ToolMessage(
        content='提取完成',
        name='personInfo',
        id='1fd9c912-1506-4673-83d2-9b74836d8571',
        tool_call_id='call_00_8gdxbIVpvChlvHDtoDg64917'
    )
]

In [24]:
#也可以使用stream模式
#包含values,tasks,messages,updates,debug,checkpoints,custom这几种模式(具体自行查找)
#多个模式可以混合使用
myagent=create_agent(
    model=model,
)
messages=HumanMessage('今年世界杯在哪里举办由谁举办？')
for stream_mode,chunk in myagent.stream({
    'messages':messages,
},
stream_mode=['values','tasks']
):
    print(f'当前模式{stream_mode}输出：{chunk}')
    print("-" * 50)

当前模式values输出：{'messages': [HumanMessage(content='今年世界杯在哪里举办由谁举办？', additional_kwargs={}, response_metadata={}, id='22e7561c-4bb5-41f8-a416-fdf058547049')]}
--------------------------------------------------
当前模式tasks输出：{'id': '22044be8-c2d4-f9e9-2d65-c1f030a26582', 'name': 'model', 'input': {'messages': [HumanMessage(content='今年世界杯在哪里举办由谁举办？', additional_kwargs={}, response_metadata={}, id='22e7561c-4bb5-41f8-a416-fdf058547049')]}, 'triggers': ('branch:to:model',)}
--------------------------------------------------
当前模式tasks输出：{'id': '22044be8-c2d4-f9e9-2d65-c1f030a26582', 'name': 'model', 'error': None, 'result': {'messages': [AIMessage(content='2026年世界杯将由**美国、加拿大和墨西哥**联合举办。这是世界杯历史上首次由三个国家共同主办，也是首次由国际足联（FIFA）旗下的三个不同足联（中北美及加勒比海足联）成员国联合承办。赛事将在三个国家的16个城市举行，其中美国11个、加拿大2个、墨西哥3个。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 12, 'total_tokens': 89, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens